In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time
import logging  
from config import ROUTES, PipelineConfig  

## CVM - Fundos Imobiliarios

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
BASE_URL     = "https://dados.cvm.gov.br/dados/FII/DOC/INF_MENSAL/DADOS/"
RAW_PATH     = f"{ROUTES.RAW_PATH}/cvm_fii/"
PATTERN_ZIP  = re.compile(r"inf_mensal_fii_(\d{4})\.zip")      
PATTERN_CSV  = re.compile(r"inf_mensal_fii_(ativo_passivo|complemento|geral)_\d{4}\.csv")
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

### 1. Verificando os arquivos

In [0]:
def arquivos(response, pattern):

    files = list()
    soup = BeautifulSoup(response, "html.parser")

    for link in soup.find_all("a", href=True):
        href = link["href"]
        match = pattern.match(href)
        if match:
            files.append(urljoin(BASE_URL, href))

    return files



In [0]:
response = PipelineConfig.retorno_text_url(url=BASE_URL)

files = arquivos(response, PATTERN_ZIP)

print(files)

### 2. Extraindo os arquivos

In [0]:
for zip_url in files: 
    log.info(f"Processando: {zip_url}")
    PipelineConfig.baixar_e_extrair_zip(url=zip_url, raw_path=RAW_PATH)

### 3. Salvar em camada Bronze Particionada

In [0]:
# ==========================================
# SCHEMA: ATIVO PASSIVO
# ==========================================
SCHEMA_ATIVO_PASSIVO = t.StructType([
    t.StructField("CNPJ_Fundo_Classe", t.StringType(), True),
    t.StructField("Data_Referencia", t.StringType(), True),
    t.StructField("Versao", t.StringType(), True),
    t.StructField("Total_Necessidades_Liquidez", t.StringType(), True),
    t.StructField("Disponibilidades", t.StringType(), True),
    t.StructField("Titulos_Publicos", t.StringType(), True),
    t.StructField("Titulos_Privados", t.StringType(), True),
    t.StructField("Fundos_Renda_Fixa", t.StringType(), True),
    t.StructField("Total_Investido", t.StringType(), True),
    t.StructField("Direitos_Bens_Imoveis", t.StringType(), True),
    t.StructField("Terrenos", t.StringType(), True),
    t.StructField("Imoveis_Renda_Acabados", t.StringType(), True),
    t.StructField("Imoveis_Renda_Construcao", t.StringType(), True),
    t.StructField("Imoveis_Venda_Acabados", t.StringType(), True),
    t.StructField("Imoveis_Venda_Construcao", t.StringType(), True),
    t.StructField("Outros_Direitos_Reais", t.StringType(), True),
    t.StructField("Acoes", t.StringType(), True),
    t.StructField("Debentures", t.StringType(), True),
    t.StructField("Bonus_Subscricao", t.StringType(), True),
    t.StructField("Certificados_Deposito_Valores_Mobiliarios", t.StringType(), True),
    t.StructField("Cedulas_Debentures", t.StringType(), True),
    t.StructField("Fundo_Acoes", t.StringType(), True),
    t.StructField("FIP", t.StringType(), True),
    t.StructField("FII", t.StringType(), True),
    t.StructField("FDIC", t.StringType(), True),
    t.StructField("Outras_Cotas_FI", t.StringType(), True),
    t.StructField("Notas_Promissorias", t.StringType(), True),
    t.StructField("Acoes_Sociedades_Atividades_FII", t.StringType(), True),
    t.StructField("Cotas_Sociedades_Atividades_FII", t.StringType(), True),
    t.StructField("CEPAC", t.StringType(), True),
    t.StructField("CRI", t.StringType(), True),
    t.StructField("CRI_CRA", t.StringType(), True),
    t.StructField("Letras_Hipotecarias", t.StringType(), True),
    t.StructField("LCI", t.StringType(), True),
    t.StructField("LCI_LCA", t.StringType(), True),
    t.StructField("LIG", t.StringType(), True),
    t.StructField("Outros_Valores_Mobliarios", t.StringType(), True),
    t.StructField("Valores_Receber", t.StringType(), True),
    t.StructField("Contas_Receber_Aluguel", t.StringType(), True),
    t.StructField("Contas_Receber_Venda_Imoveis", t.StringType(), True),
    t.StructField("Outros_Valores_Receber", t.StringType(), True),
    t.StructField("Rendimentos_Distribuir", t.StringType(), True),
    t.StructField("Taxa_Administracao_Pagar", t.StringType(), True),
    t.StructField("Taxa_Performance_Pagar", t.StringType(), True),
    t.StructField("Obrigacoes_Aquisicao_Imoveis", t.StringType(), True),
    t.StructField("Adiantamento_Venda_Imoveis", t.StringType(), True),
    t.StructField("Adiantamento_Alugueis", t.StringType(), True),
    t.StructField("Obrigacoes_Securitizacao_Recebiveis", t.StringType(), True),
    t.StructField("Instrumentos_Financeiros_Derivativos", t.StringType(), True),
    t.StructField("Provisoes_Contigencias", t.StringType(), True),
    t.StructField("Outros_Valores_Pagar", t.StringType(), True),
    t.StructField("Total_Passivo", t.StringType(), True),
])

# ==========================================
# SCHEMA: COMPLEMENTO
# ==========================================
SCHEMA_COMPLEMENTO = t.StructType([
    t.StructField("CNPJ_Fundo_Classe", t.StringType(), True),
    t.StructField("Data_Referencia", t.StringType(), True),
    t.StructField("Versao", t.StringType(), True),
    t.StructField("Data_Informacao_Numero_Cotistas", t.StringType(), True),
    t.StructField("Total_Numero_Cotistas", t.StringType(), True),
    t.StructField("Numero_Cotistas_Pessoa_Fisica", t.StringType(), True),
    t.StructField("Numero_Cotistas_Pessoa_Juridica_Nao_Financeira", t.StringType(), True),
    t.StructField("Numero_Cotistas_Banco_Comercial", t.StringType(), True),
    t.StructField("Numero_Cotistas_Corretora_Distribuidora", t.StringType(), True),
    t.StructField("Numero_Cotistas_Outras_Pessoas_Juridicas_Financeira", t.StringType(), True),
    t.StructField("Numero_Cotistas_Investidores_Nao_Residentes", t.StringType(), True),
    t.StructField("Numero_Cotistas_Entidade_Aberta_Previdencia_Complementar", t.StringType(), True),
    t.StructField("Numero_Cotistas_Entidade_Fechada_Previdencia_Complementar", t.StringType(), True), # <-- Corrigido o caractere quebrado aqui
    t.StructField("Numero_Cotistas_Regime_Proprio_Previdencia_Servidores_Publicos", t.StringType(), True),
    t.StructField("Numero_Cotistas_Sociedade_Seguradora_Resseguradora", t.StringType(), True),
    t.StructField("Numero_Cotistas_Sociedade_Capitalizacao_Arrendamento_Mercantil", t.StringType(), True),
    t.StructField("Numero_Cotistas_FII", t.StringType(), True),
    t.StructField("Numero_Cotistas_Outros_Fundos", t.StringType(), True),
    t.StructField("Numero_Cotistas_Distribuidores_Fundo", t.StringType(), True),
    t.StructField("Numero_Cotistas_Outros_Tipos", t.StringType(), True),
    t.StructField("Valor_Ativo", t.StringType(), True),
    t.StructField("Patrimonio_Liquido", t.StringType(), True),
    t.StructField("Cotas_Emitidas", t.StringType(), True),
    t.StructField("Valor_Patrimonial_Cotas", t.StringType(), True),
    t.StructField("Percentual_Despesas_Taxa_Administracao", t.StringType(), True),
    t.StructField("Percentual_Despesas_Agente_Custodiante", t.StringType(), True),
    t.StructField("Percentual_Rentabilidade_Efetiva_Mes", t.StringType(), True),
    t.StructField("Percentual_Rentabilidade_Patrimonial_Mes", t.StringType(), True),
    t.StructField("Percentual_Dividend_Yield_Mes", t.StringType(), True),
    t.StructField("Percentual_Amortizacao_Cotas_Mes", t.StringType(), True),
])

# ==========================================
# SCHEMA: GERAL
# ==========================================
SCHEMA_GERAL = t.StructType([
    t.StructField("Tipo_Fundo_Classe", t.StringType(), True),
    t.StructField("CNPJ_Fundo_Classe", t.StringType(), True),
    t.StructField("Data_Referencia", t.StringType(), True),
    t.StructField("Versao", t.StringType(), True),
    t.StructField("Data_Entrega", t.StringType(), True),
    t.StructField("Nome_Fundo_Classe", t.StringType(), True),
    t.StructField("Data_Funcionamento", t.StringType(), True),
    t.StructField("Publico_Alvo", t.StringType(), True),
    t.StructField("Codigo_ISIN", t.StringType(), True),
    t.StructField("Quantidade_Cotas_Emitidas", t.StringType(), True),
    t.StructField("Fundo_Exclusivo", t.StringType(), True),
    t.StructField("Cotistas_Vinculo_Familiar", t.StringType(), True),
    t.StructField("Mandato", t.StringType(), True),
    t.StructField("Segmento_Atuacao", t.StringType(), True),
    t.StructField("Tipo_Gestao", t.StringType(), True),
    t.StructField("Prazo_Duracao", t.StringType(), True),
    t.StructField("Data_Prazo_Duracao", t.StringType(), True),
    t.StructField("Encerramento_Exercicio_Social", t.StringType(), True),
    t.StructField("Mercado_Negociacao_Bolsa", t.StringType(), True),
    t.StructField("Mercado_Negociacao_MBO", t.StringType(), True),
    t.StructField("Mercado_Negociacao_MB", t.StringType(), True),
    t.StructField("Entidade_Administradora_BVMF", t.StringType(), True),
    t.StructField("Entidade_Administradora_CETIP", t.StringType(), True),
    t.StructField("Nome_Administrador", t.StringType(), True),
    t.StructField("CNPJ_Administrador", t.StringType(), True),
    t.StructField("Logradouro", t.StringType(), True),
    t.StructField("Numero", t.StringType(), True),
    t.StructField("Complemento", t.StringType(), True),
    t.StructField("Bairro", t.StringType(), True),
    t.StructField("Cidade", t.StringType(), True),
    t.StructField("Estado", t.StringType(), True),
    t.StructField("CEP", t.StringType(), True),
    t.StructField("Telefone1", t.StringType(), True),
    t.StructField("Telefone2", t.StringType(), True),
    t.StructField("Telefone3", t.StringType(), True),
    t.StructField("Site", t.StringType(), True),
    t.StructField("Email", t.StringType(), True),
])

In [0]:
# Dicionário para organizar arquivos por tipo
lista_registros = {
    "fii_ativo_passivo_cvm": [],
    "fii_complemento_cvm": [],
    "fii_geral_cvm": []
}

# Lista arquivos da pasta RAW
arquivos = dbutils.fs.ls(RAW_PATH)

# Classifica os arquivos
for file in arquivos:
    nome_arquivo = os.path.basename(file.path)
    match = PATTERN_CSV.match(nome_arquivo)

    if match:
        tipo = match.group(1)
        if tipo == "ativo_passivo":
            lista_registros["fii_ativo_passivo_cvm"].append(file.path)
        elif tipo == "complemento":
            lista_registros["fii_complemento_cvm"].append(file.path)
        elif tipo == "geral":
            lista_registros["fii_geral_cvm"].append(file.path)


for name_path, caminhos in lista_registros.items():

    if not caminhos:
        continue

    bronze_path = f"{ROUTES.TABLE_BASE}.bronze_{name_path}"

    print(f"\nProcessamento: {name_path}")
    print(f"Arquivos: {len(caminhos)}")
    print(f"Output: {bronze_path}")

    # =========================================================
    # SOLUÇÃO PARA O SCHEMA DRIFT NO ARQUIVO GERAL
    # =========================================================
    if name_path == "fii_geral_cvm":
        arquivos_antigo = []
        arquivos_novo = []

        # 1. Espiada Rápida (Separa os arquivos pelo layout)
        for path in caminhos:
            colunas = spark.read.option("sep", ";").option("header", "true").csv(path).columns
            # Se tiver CNPJ_Fundo, sabemos que é o layout antigo de 2016
            if "CNPJ_Fundo" in colunas:
                arquivos_antigo.append(path)
            else:
                arquivos_novo.append(path)

        dfs_para_unir = []

        # 2. Leitura em Lote: Layout Novo
        if arquivos_novo:
            # Aqui você aplica o SCHEMA_GERAL que te passei na resposta anterior!
            df_novos = (spark.read
                .schema(SCHEMA_GERAL) 
                .option("encoding", "ISO-8859-1")
                .option("sep", ";")
                .option("header", "true")
                .csv(arquivos_novo)
            )
            dfs_para_unir.append(df_novos)

        # 3. Leitura em Lote e Correção: Layout Antigo
        if arquivos_antigo:
            # Lemos o bloco antigo inteiro de uma vez
            df_antigos = (spark.read
                .option("encoding", "ISO-8859-1")
                .option("sep", ";")
                .option("header", "true")
                .csv(arquivos_antigo)
            )
            
            # Aplicamos as regras de correção em massa para todo o bloco legado
            df_antigos = (df_antigos
                .withColumnRenamed("CNPJ_Fundo", "CNPJ_FUNDO_CLASSE")
                .withColumnRenamed("Nome_Fundo", "Nome_Fundo_Classe")
                .withColumn("Tipo_Fundo_Classe", f.lit(None).cast("string"))
            )
            dfs_para_unir.append(df_antigos)

        # 4. O Único Union Necessário (O cluster agradece!)
        if len(dfs_para_unir) == 2:
            df = dfs_para_unir[0].unionByName(dfs_para_unir[1], allowMissingColumns=True)
        else:
            df = dfs_para_unir[0]

    # =========================================================
    # LEITURA DOS OUTROS ARQUIVOS (Sem Schema Drift)
    # =========================================================
    else:
        # Seleciona o schema correto dependendo do nome no dicionário
        schema_atual = SCHEMA_ATIVO_PASSIVO if name_path == "fii_ativo_passivo_cvm" else SCHEMA_COMPLEMENTO
        
        # Leitura padrão em lote numa tacada só
        df = (spark.read
            .schema(schema_atual)
            .option("encoding", "ISO-8859-1")
            .option("sep", ";")
            .option("header", "true")
            .csv(caminhos)
        )
        
    # =========================================================
    # METADADOS E ESCRITA (Igual para todos)
    # =========================================================
    df = (df
        .withColumn("_source_url", f.lit(BASE_URL))
        .withColumn("_ingest_timestamp", f.current_timestamp())
        .withColumn("data_processamento", f.lit(DATA_PROC))
    )


    try:
        # 7. Escrita na Bronze 
        n = df.count()

        log.info(f"Escrevendo {n} linhas em Bronze ")

        (df.write 
            .mode("overwrite") 
            .option("replaceWhere", f"data_processamento = {DATA_PROC}") 
            .option("mergeSchema", "true")
            .option("delta.autoOptimize.optimizeWrite", "true")
            .option("delta.autoOptimize.autoCompact", "true")
            .partitionBy("data_processamento") 
            .format("delta") 
            .saveAsTable(bronze_path)
        )

        PipelineConfig.registrar_auditoria(
            spark, ROUTES.AUDIT_PATH, "bronze_raw_cvm_fundos_imobliarios",
            bronze_path, n, "SUCESSO", DATA_PROC
        )

    except Exception as e:
        PipelineConfig.registrar_auditoria(
            spark, ROUTES.AUDIT_PATH, "bronze_raw_cvm_fundos_imobliarios",
            bronze_path, 0, "FALHA", DATA_PROC, str(e)
        )
        raise